# MemPrimitive 功能展示

## 1. MemPrimitive方法的pipeline

在 `MemPrimitive` 里，memory system 不是黑盒方法名，而是一条可以逐段替换的流水线：

```text
ingest:
  unit_formation
  -> representation
  -> write_trigger
  -> organization
  -> evolution_trigger
  -> memory_evolution

recall:
  retrieval
  -> readout
```

下面的单元格会依次展示：

- 最小可用 memory module
- 多层 topology
- 可组合 trigger
- dispatch fan-out
- 适合继续运行的 demonstration 脚本

In [ ]:
from pathlib import Path
import sys
from pprint import pprint

repo_root = Path.cwd()
if not (repo_root / "memprimitive").exists():
    for candidate in [repo_root, *repo_root.parents]:
        if (candidate / "memprimitive").exists():
            repo_root = candidate
            break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"repo_root = {repo_root}")

In [ ]:
from memprimitive import (
    DispatchOrganization,
    MemoryPipeline,
    MemoryStore,
    Observation,
    Packet,
    Query,
    StoreLayerSpec,
    StoreTopology,
)
from memprimitive.baselines import (
    AlwaysWriteTrigger,
    AppendOnlyEvolution,
    AppendOrganization,
    BasicRepresentation,
    ConcatenateReadout,
    EntityRetrieval,
    GraphAppendOrganization,
    LayerAwareRetrieval,
    PassThroughUnitFormation,
    RecencyRetrieval,
)
from memprimitive.baselines._trigger_family import (
    AlwaysOpenGate,
    BooleanGatePolicy,
    ConstantSignal,
    ThresholdPolicy,
    WeightedSumScorer,
)
from memprimitive.baselines.evolution_trigger import compose_evolution_trigger
from memprimitive.baselines.write_trigger import compose_write_trigger

## 2. 最小可用 memory module

先拼出一个最小闭环，再逐个替换 slot。

In [ ]:
minimal_pipeline = MemoryPipeline(
    unit_formation=PassThroughUnitFormation(),
    representation=BasicRepresentation(),
    write_trigger=AlwaysWriteTrigger(),
    organization=AppendOrganization(),
    retrieval=RecencyRetrieval(top_k=2),
    readout=ConcatenateReadout(),
)

minimal_pipeline

In [ ]:
minimal_pipeline.ingest(Observation(text="The user likes concise examples.", source="dialogue"))
minimal_pipeline.ingest(Observation(text="The user works on compositional memory.", source="notes"))

minimal_readout = minimal_pipeline.recall(Query(text="What does the user like?"))
print(minimal_readout.text)
print("source_ids:", minimal_readout.source_ids)

可以在上面的代码里直接替换某个 slot，比如：

- 把 `AlwaysWriteTrigger()` 换成组合 trigger
- 把 `RecencyRetrieval(top_k=2)` 换成别的 retrieval
- 把 `AppendOrganization()` 换成分层路由或 graph organization

这就是 `MemPrimitive` 灵活性的核心来源。

## 3. topology：结构是声明出来的

多层 memory 不是另起一套系统，而是 `StoreTopology` 的一部分。

In [ ]:
topology = StoreTopology.from_layers(
    [
        StoreLayerSpec(name="working", theme="working", indices=("temporal", "keyword")),
        StoreLayerSpec(name="episodic", theme="session_memory", indices=("temporal", "keyword")),
        StoreLayerSpec(
            name="knowledge_graph",
            theme="knowledge_graph",
            shape="Graph",
            indices=("graph", "entity"),
        ),
    ]
)
store = MemoryStore(topology=topology)

topology_pipeline = MemoryPipeline(
    unit_formation=PassThroughUnitFormation(),
    representation=BasicRepresentation(),
    write_trigger=AlwaysWriteTrigger(),
    organization=AppendOrganization(target_layer="episodic"),
    retrieval=RecencyRetrieval(top_k=2, layer="episodic"),
    readout=ConcatenateReadout(),
    store=store,
)

packet = topology_pipeline.ingest(
    Observation(text="The user wants a store with an explicit topology.", source="notes")
)
topology_pipeline.ingest(
    Observation(text="The episodic layer keeps recent dialogue-like memories.", source="notes")
)

pprint([
    {
        "name": layer.name,
        "theme": layer.theme,
        "shape": layer.shape,
        "indices": layer.indices,
    }
    for layer in topology_pipeline.store.topology.layers
])
print("organization target layer:", packet.trace["organization"]["target_layer"])
print("records per layer:", {name: topology_pipeline.store.count(name) for name in topology_pipeline.store.topology.layer_names})

可以修改：

- 增减 layer
- 改 `shape`
- 改 `indices`
- 改 `organization` 和 `retrieval` 指向的 layer

结构层和行为层是解耦的。

## 4. trigger可以直接组装

其触发逻辑可以拆成 signal、scorer、gate、policy 四段。

In [ ]:
trigger_pipeline = MemoryPipeline(
    unit_formation=PassThroughUnitFormation(),
    representation=BasicRepresentation(),
    write_trigger=compose_write_trigger(
        name="demo_threshold_write_trigger",
        signal_providers=(ConstantSignal(signal_name="importance_hint", value=0.8),),
        scorer=WeightedSumScorer(weights={"importance_hint": 1.0}),
        gate=AlwaysOpenGate(),
        policy=ThresholdPolicy(threshold=0.5),
    ),
    organization=AppendOrganization(),
    evolution_trigger=compose_evolution_trigger(
        name="demo_boolean_evolution_trigger",
        signal_providers=(ConstantSignal(signal_name="after_write_ready", value=1.0),),
        scorer=WeightedSumScorer(weights={"after_write_ready": 1.0}),
        gate=AlwaysOpenGate(),
        policy=BooleanGatePolicy(),
    ),
    memory_evolution=AppendOnlyEvolution(),
    retrieval=RecencyRetrieval(top_k=2),
    readout=ConcatenateReadout(),
)

trigger_packet = trigger_pipeline.ingest(
    Observation(text="The user is exploring compositional triggers.", source="notes")
)

print("write_trigger trace:")
pprint(trigger_packet.trace["write_trigger"])
print()
print("evolution_trigger trace:")
pprint(trigger_packet.trace["evolution_trigger"])


可以直接修改：

- `ConstantSignal` 的值
- `WeightedSumScorer` 的权重
- `ThresholdPolicy` 的阈值
- `BooleanGatePolicy()` 换成别的 policy

## 5. dispatch：一次 ingest 进入多个 memory view

如果想让同一份输入同时进入普通层和 graph 层，可以用 `DispatchOrganization`。

In [ ]:
dispatch_topology = StoreTopology.from_layers(
    [
        StoreLayerSpec(name="working", theme="working", indices=("temporal", "keyword")),
        StoreLayerSpec(
            name="knowledge_graph",
            theme="knowledge_graph",
            shape="Graph",
            indices=("graph", "entity"),
        ),
    ]
)
dispatch_store = MemoryStore(topology=dispatch_topology)

dispatch_pipeline = MemoryPipeline(
    representation=BasicRepresentation(elements=("text", "tags", "entities", "triple", "tags")),
    organization=DispatchOrganization(
        (
            AppendOrganization(target_layer="working"),
            GraphAppendOrganization(target_layer="knowledge_graph"),
        ),
        primary_index=0,
    ),
    retrieval=LayerAwareRetrieval(
        default_retriever=RecencyRetrieval(top_k=2),
        retriever_by_layer={"knowledge_graph": EntityRetrieval(top_k=2)},
        top_k=4,
    ),
    readout=ConcatenateReadout(separator="\n\n"),
    store=dispatch_store,
)

dispatch_pipeline.ingest(Observation(text="Alice is debugging the retrieval merge order.", source="dialogue"))
dispatch_pipeline.ingest(Observation(text="The current task is to explain graph-backed recall.", source="dialogue"))

dispatch_readout = dispatch_pipeline.recall(Query(text="Alice"))
retrieval_packet, _ = dispatch_pipeline.retrieval.run(Packet(query=Query(text="Alice")), dispatch_store)

print("records per layer:")
pprint({name: dispatch_store.count(name) for name in dispatch_store.topology.layer_names})
print()
print("layer-aware retrieval trace:")
pprint(retrieval_packet.trace["retrieval"])
print()
print(dispatch_readout.text)
print("source_ids:", dispatch_readout.source_ids)

- 同一输入可以同时进入多个 memory view
- 多层结构和 retrieval 编排仍然保持统一接口
- graph 能力不是外挂，而是 pipeline 里的正常组合方式

## 6. 其他 demonstration

下面这些脚本更适合在终端里整段运行：

- `python -m memprimitive.example.demonstration.minimal_pipeline`
- `python -m memprimitive.example.demonstration.topology_store`
- `python -m memprimitive.example.demonstration.composed_triggers`
- `python -m memprimitive.example.demonstration.dispatch_organization_recall`
- `python -m memprimitive.example.demonstration.graph_baseline_pipeline`
- `python -m memprimitive.example.demonstration.graph_dependent_pipeline`
- `python -m memprimitive.example.demonstration.reflexion_reflection_cycle`
- `python -m memprimitive.example.demonstration.amem_like_graph_cycle`

其中最后两个需要真实 LLM 相关环境变量：

- `MEMPRIMITIVE_API_KEY`
- `MEMPRIMITIVE_BASE_URL`
- `MEMPRIMITIVE_MODEL`